In [ ]:
import ipywidgets as W
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Javascript, display

from brand_kit.theme import FONT, GRAY, GRID, NAVY, ORANGE, TEXT, TEXT_SOFT, WHITE

df = pd.read_csv("ai_job_dataset.csv")
LEVELS = ["Entry", "Mid", "Senior", "Executive"]
SIZES = ["Small", "Medium", "Large"]
df["experience"] = df["experience_level"].map(dict(zip(["EN", "MI", "SE", "EX"], LEVELS)))
df["size"] = df["company_size"].map(dict(zip("SML", SIZES)))
assert len(df) == 15000 and not df[["experience", "size"]].isna().any().any()

entry = df[df["experience"] == "Entry"]          # chapters 1 and 2 are about a first job
START = entry["salary_usd"].median()
entry_skills = entry["required_skills"].str.get_dummies(sep=", ").astype(bool)
pay = entry["salary_usd"].to_numpy(float)

the = lambda c: f"the {c}" if c.startswith(("United", "Nether")) else c
money = lambda v: f"${v / 1000:,.0f}k"
def delta(v):
    return f"{'−' if v < 0 else '+'}${abs(v) / 1000:,.1f}k" if abs(v) < 10000 else f"{'−' if v < 0 else '+'}${abs(v) / 1000:,.0f}k"

def r_squared(y, X):
    """Adjusted R2: share of variation in y explained by dummy columns X (0-1)."""
    X = np.column_stack([np.ones(len(y)), X])
    resid = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
    return max(0.0, 1 - (resid @ resid / (len(y) - np.linalg.matrix_rank(X))) / y.var(ddof=1))

# Which rows belong to each skill / role / industry
masks = {
    "Skill": {s: entry_skills[s].to_numpy() for s in sorted(entry_skills)},
    "Role": {t: (entry["job_title"] == t).to_numpy() for t in sorted(entry["job_title"].unique())},
    "Industry": {i: (entry["industry"] == i).to_numpy() for i in sorted(entry["industry"].unique())},
}
medians = lambda y, group: pd.Series({k: np.median(y[m]) for k, m in group.items()})
focus = {name: medians(pay, g).sort_values(ascending=False) for name, g in masks.items()}

# How big a top-to-bottom gap does pure chance produce? Shuffle salaries 200 times and see.
rng = np.random.default_rng(0)
shuffles = [rng.permutation(pay) for _ in range(200)]
chance, p_value = {}, {}
for name, g in masks.items():
    sim = np.array([np.ptp(medians(y, g).to_numpy()) for y in shuffles])
    chance[name] = np.median(sim)
    p_value[name] = np.mean(sim >= np.ptp(focus[name].to_numpy()))

# Static insight numbers
dummies = lambda d, c: pd.get_dummies(d[c]).to_numpy(float)
all_focus = np.column_stack([entry_skills.to_numpy(float), dummies(entry, "job_title"), dummies(entry, "industry")])
share = {"Country": r_squared(pay, dummies(entry, "company_location")),
         "Company size": r_squared(pay, dummies(entry, "size")),
         "Skill, role, industry": r_squared(pay, all_focus)}
by_country = entry.groupby("company_location")["salary_usd"].median().sort_values(ascending=False)
by_level = df.groupby("experience")["salary_usd"].median()[LEVELS]

# The written story depends on these staying true
assert share["Country"] > share["Company size"] > 0.1 and share["Skill, role, industry"] < 0.01
assert min(p_value.values()) > 0.05, p_value

In [ ]:
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Libre+Franklin:wght@400;600;800&display=swap');
:root { --navy:#232D4B; --gray:#C9CBD2; --orange:#E57200; --peach:#F9DCBE; --soft:#4A5677; --page:#F6F6F8; --line:#E4E5EA;
  --jp-widgets-slider-active-handle-color:var(--navy); --jp-widgets-input-focus-border-color:var(--orange); --gap:12px; }
html, body, #rendered_cells { background:var(--page) !important; }
#rendered_cells, #rendered_cells .jp-Cell, #rendered_cells .jp-OutputArea-output { padding:0 !important; margin:0 !important; }
.jp-OutputPrompt, .jp-InputPrompt { display:none !important; }
.jp-OutputArea-child { table-layout:fixed !important; width:100% !important; }
.story, .story .jp-RenderedHTMLCommon, .story .widget-html-content, .story .widget-label, .story .widget-readout,
.story .widget-button, .story select { font-family:'Libre Franklin','Franklin Gothic',Arial,sans-serif !important; }
.story .widget-html, .story .widget-html-content { margin:0 !important; line-height:1.4 !important; font-size:14px !important; color:var(--navy); height:auto !important; min-height:0; }
.story .widget-html-content { width:100%; }

/* One screen: navy opening on the left, three steps, then the result underneath */
.story { position:fixed !important; inset:0; display:grid !important; overflow:hidden !important; background:var(--page);
  grid-template-columns:minmax(270px, 21vw) repeat(3, minmax(0, 1fr)); grid-template-rows:minmax(0, 1fr) auto; gap:var(--gap);
  padding:var(--gap) var(--gap) var(--gap) 0; box-sizing:border-box; z-index:5; }
.story > * { margin:0 !important; min-width:0; min-height:0; max-width:none; }
.story > .opening { grid-row:1 / span 2; background:var(--navy); color:#fff; margin:calc(-1 * var(--gap)) 0 calc(-1 * var(--gap)) !important;
  border-right:8px solid var(--orange); padding:26px 24px 18px; overflow-y:auto !important; box-sizing:border-box; }
.opening .widget-html-content, .opening p, .opening li { color:#fff; }
.opening h1 { font-size:30px !important; line-height:1.12 !important; font-weight:800 !important; color:#fff !important; margin:0 0 12px !important; text-wrap:balance; letter-spacing:-.01em; }
.opening .hook { font-size:16px; line-height:1.45; color:var(--peach) !important; margin:0 0 14px !important; }
.opening h2 { font-size:15px !important; font-weight:800 !important; color:#fff !important; margin:0 0 4px !important; }
.opening p { margin:0 0 12px !important; line-height:1.45; }
.opening ol { margin:0 0 12px !important; padding-left:18px !important; }
.opening li { margin-bottom:3px !important; }
.opening a { color:#fff !important; text-decoration:underline !important; text-decoration-color:var(--orange) !important; text-underline-offset:3px; }
.opening .foot, .opening .by { font-size:12px; line-height:1.4; color:var(--gray) !important; margin:0 0 8px !important; }

.step { background:#fff; border:1px solid var(--line); border-radius:8px; padding:14px 16px 12px; display:flex; flex-direction:column; overflow:hidden !important; box-sizing:border-box; }
.step > * { flex:none; max-width:100%; }
.step h2 { font-size:20px !important; line-height:1.2 !important; font-weight:800 !important; color:var(--navy) !important; margin:0 !important; display:flex; align-items:center; gap:10px; }
.step h2 i { font-style:normal; flex:none; width:28px; height:28px; border-radius:50%; background:var(--navy); color:#fff; font-size:15px; display:grid; place-items:center; }
.step .lead { margin:4px 0 6px !important; color:var(--soft); }
.step .say { margin:4px 0 0 !important; font-size:15px; min-height:2.7em; }
.step .say b { font-weight:800; }
.plot { flex:1 1 0 !important; min-height:110px; width:100%; overflow:hidden; }
.plot > div, .plot .js-plotly-plot, .plot .plot-container { height:100% !important; }
.step .widget-inline-hbox { margin:2px 0 !important; width:100% !important; }
.step .widget-label { font-size:13px; color:var(--soft); width:auto !important; min-width:92px; text-align:left; }
.step .widget-readout { font-size:13px; color:var(--soft); }
.step select { color:var(--navy) !important; border:1px solid var(--gray) !important; border-radius:6px; background-color:#fff !important; font-size:14px; }
.step .widget-toggle-buttons { margin:2px 0 !important; }
.step .widget-toggle-button { background:var(--line) !important; color:var(--navy) !important; border-radius:6px; box-shadow:none !important; font-weight:600; }
.step .widget-toggle-button.mod-active { background:var(--navy) !important; color:#fff !important; }
.story button:focus-visible, .story select:focus-visible, .story a:focus-visible { outline:3px solid var(--orange) !important; outline-offset:1px; }
.story .noUi-connect { background:var(--navy) !important; } .story .noUi-handle { border-color:var(--navy) !important; }

.insight { border-top:1px solid var(--line); padding-top:8px; margin-top:4px; }
.insight b { display:block; font-weight:800; margin-bottom:4px; }
.hb { display:grid; grid-template-columns:132px 1fr; align-items:center; gap:8px; font-size:12px; line-height:17px; }
.hb > span:last-child { display:flex; align-items:center; gap:6px; font-weight:600; padding-right:42px; }
.hb i { display:block; flex:none; height:11px; border-radius:0 3px 3px 0; min-width:2px; }

.result { grid-column:2 / span 3; background:var(--peach); border-radius:8px; padding:12px 18px; overflow:hidden !important; }
.res { display:grid; grid-template-columns:1.35fr repeat(3, .8fr) 1.25fr; gap:20px; align-items:start; color:var(--navy); }
.res p { margin:0 !important; line-height:1.35; }
.res .path { font-size:15px; } .res .path b, .res .ans b { display:block; font-size:18px; font-weight:800; line-height:1.2; margin-bottom:3px; }
.res .n b { display:block; font-size:28px; line-height:1.05; font-weight:800; white-space:nowrap; }
.res .n span { font-size:13px; line-height:1.3; display:block; }

@media (max-height:860px) and (min-width:1101px) {
  :root { --gap:8px; }
  .story > .opening { padding:16px 18px 10px; } .opening h1 { font-size:25px !important; margin-bottom:8px !important; } .opening .hook { font-size:14px; margin-bottom:8px !important; }
  .opening p, .opening ol, .opening li { font-size:13px; line-height:1.36; } .opening p, .opening ol { margin-bottom:7px !important; } .opening .foot, .opening .by { font-size:11px; margin-bottom:5px !important; }
  .step { padding:9px 12px 8px; } .step h2 { font-size:18px !important; } .step .lead { margin:2px 0 3px !important; font-size:13px; } .step .say { font-size:14px; margin-top:2px !important; }
  .insight { padding-top:5px; margin-top:2px; } .insight b { margin-bottom:1px; font-size:13px; } .hb { line-height:14px; } .hb i { height:9px; }
  .result { padding:8px 14px; } .res .n b { font-size:24px; }
}
@media (min-width:1101px) and (max-height:640px), (min-width:1101px) and (max-width:1320px) and (max-height:720px) {
  .opening h1 { font-size:22px !important; } .opening .hook { font-size:13px; line-height:1.36; }
  .opening p, .opening ol, .opening li { font-size:12px; line-height:1.3; } .opening p, .opening ol { margin-bottom:5px !important; } .opening h2 { font-size:13px !important; margin-bottom:1px !important; }
  .opening .foot, .opening .by { font-size:10.5px; line-height:1.3; }
}
@media (max-width:1100px), (max-height:560px) {
  .story { position:static !important; grid-template-columns:1fr; grid-template-rows:none; padding:0 0 12px; overflow:visible !important; }
  .opening, .result { grid-column:1; grid-row:auto; } .story > .opening { margin:0 !important; border-right:0; border-bottom:8px solid var(--orange); }
  .story > .step, .story > .result { margin:0 12px !important; } .plot { flex:none !important; height:280px; } .res { grid-template-columns:repeat(3, 1fr); } .res .path, .res .ans { grid-column:1 / -1; }
}
</style>"""))

In [ ]:
def static_bars(title, values, fmt, hot):
    """Static bars (no interaction): the insight each step supports."""
    top = max(values.values())
    rows = "".join(f'<div class="hb"><span>{k}</span><span><i style="width:{v / top:.1%};background:{ORANGE if k == hot else GRAY}"></i>{fmt(v)}</span></div>'
                   for k, v in values.items())
    return W.HTML(f'<div class="insight"><b>{title}</b>{rows}</div>')

pct = lambda v: "<1%" if v < .01 else f"{v:.0%}"
best, worst = by_country.index[0], by_country.index[-1]
insight1 = static_bars("Skill, role and industry explain almost none of entry-level pay", share, pct, "Skill, role, industry")
insight2 = static_bars(f"{best} starts at {by_country.iloc[0] / by_country.iloc[-1]:.1f} times {worst}'s pay",
                       {c: by_country[c] for c in [best, "United States", worst]}, money, best)
insight3 = static_bars(f"Executives earn {by_level['Executive'] / by_level['Entry']:.1f} times entry-level pay",
                       {k: by_level[k] for k in ["Entry", "Senior", "Executive"]}, money, "Executive")

opening = W.HTML("""
<h1>What does the AI job market reward most?</h1>
<p class="hook">You're about to graduate and want the best-paid AI job. Should you chase a skill, a place or a promotion? Choose in each step and watch your pay change.</p>
<h2>Our goal</h2>
<p>Help students decide where to put their effort by answering three questions:</p>
<ol><li>Does the skill, role or industry you choose change your starting pay?</li><li>Does where you apply change it?</li><li>How does pay grow with experience?</li></ol>
<h2>The data*</h2>
<p><a href="https://www.kaggle.com/datasets/bismasajjad/global-ai-job-market-and-salary-trends-2025" target="_blank" rel="noopener">Global AI Job Market &amp; Salary Trends 2025</a> by Bisma Sajjad on Kaggle (CC0). 15,000 AI job postings from 20 countries, January 2024 to April 2025. We use salary (USD), experience, skills, job title, industry, country and company size.</p>
<p class="foot">* This is a synthetic dataset created for educational purposes to simulate AI job market patterns. All data is algorithmically generated based on industry research and market trends.</p>
<p class="by">By #unemployed: Michael, Brandon, Mia, Hassan, Jafar</p>""")
opening.add_class("opening")

In [ ]:
# ---------- choices ----------
full = W.Layout(width="100%", min_width="0")
c_kind = W.ToggleButtons(options=list(masks), value="Skill", style=W.ToggleButtonsStyle(button_width="80px"))
c_pick = W.Dropdown(options=list(masks["Skill"]), value="Python", description="Your skill", layout=full)
c_country = W.Dropdown(options=sorted(by_country.index), value="United States", description="Country", layout=full)
c_size = W.Dropdown(options=["Any size"] + SIZES, description="Company", layout=full)
c_years = W.IntSlider(value=10, min=0, max=19, description="Years worked", layout=full)
say1, say2, say3, result = W.HTML(), W.HTML(), W.HTML(), W.HTML()
result.add_class("result")

# ---------- charts ----------
X_MAX = 140000  # steps 1 and 2 share one scale, so bar lengths compare across charts

def make_fig():
    f = go.Figure()
    f.update_layout(template="none", autosize=True, margin=dict(l=4, r=4, t=6, b=4), showlegend=False, dragmode=False, bargap=0.38,
                    font=dict(family=FONT, size=12, color=TEXT), paper_bgcolor=WHITE, plot_bgcolor=WHITE,
                    hoverlabel=dict(bgcolor="white", font_size=13))
    f.update_xaxes(gridcolor=GRID, zeroline=False, automargin=True, fixedrange=True)
    f.update_yaxes(gridcolor=GRID, zeroline=False, automargin=True, fixedrange=True, ticks="")
    return f

def make_bars(hover):
    f = make_fig()
    f.add_bar(orientation="h", hovertemplate=hover + "<extra></extra>")
    f.add_scatter(mode="text", textposition="middle left", hoverinfo="skip", textfont=dict(size=13, color=TEXT))
    f.update_xaxes(visible=False, range=[0, X_MAX])
    f.update_yaxes(showgrid=False, dtick=1, autorange="reversed", ticksuffix=" ")
    f.add_vline(x=START, line=dict(color=NAVY, width=1))
    f.add_annotation(x=START, y=1, yref="paper", yanchor="bottom", text=f"typical start {money(START)}", showarrow=False, font=dict(size=11, color=TEXT_SOFT))
    f.update_layout(margin_t=20)
    return f

fig1 = make_bars("<b>%{y}</b><br>Median entry-level pay %{x:$,.0f}<br>%{customdata:,} postings<br>Click to choose")
fig2 = make_bars("<b>%{y}</b><br>Median entry-level pay %{x:$,.0f}<br>%{customdata:,} postings<br>Click to choose")
fig3 = make_fig()
fig3.add_scatter(mode="lines", line=dict(color=GRAY, width=3, shape="hv"), hovertemplate="All countries, year %{x}<br>%{y:$,.0f}<extra></extra>")
fig3.add_scatter(mode="lines", line=dict(color=NAVY, width=3, shape="hv"), hovertemplate="Your place, year %{x}<br>%{y:$,.0f}<extra></extra>")
fig3.add_scatter(mode="markers+text", marker=dict(color=ORANGE, size=14, line=dict(color=WHITE, width=2)), textfont=dict(size=15, color=TEXT), hoverinfo="skip", cliponaxis=False)
fig3.update_xaxes(title="Years of experience", range=[-.6, 19.8], tickvals=[0, 2, 5, 10, 15, 19], showgrid=False)
fig3.update_yaxes(tickprefix="$", tickformat="~s", nticks=5)
fig3.add_scatter(mode="text", x=[19.6, 19.6], textposition="bottom left", hoverinfo="skip", cliponaxis=False, textfont=dict(size=12, color=TEXT))
fig3.update_layout(margin_t=22)

YEAR_STEPS = [0, 2, 5, 10, 19]                    # entry 0-1, mid 2-4, senior 5-9, executive 10-19
level_at = lambda y: LEVELS[int(np.searchsorted(YEAR_STEPS[1:4], y, side="right"))]
assert [level_at(y) for y in (0, 1, 2, 4, 5, 9, 10, 19)] == ["Entry"] * 2 + ["Mid"] * 2 + ["Senior"] * 2 + ["Executive"] * 2

def place(d):
    """Rows matching the company-size choice."""
    return d if c_size.value == "Any size" else d[d["size"] == c_size.value]

def ranked(s, chosen, n=7):
    """Top n plus the lowest, always keeping the chosen one in view."""
    keep = set(s.index[:n]) | {s.index[-1], chosen}
    return s[[k for k in s.index if k in keep]]

def draw_bars(fig, s, counts, chosen):
    with fig.batch_update():
        b = fig.data[0]
        b.y, b.x, b.customdata = list(s.index), list(s.values), [int(counts[k]) for k in s.index]
        t = fig.data[1]
        t.y, t.x = list(s.index), [X_MAX] * len(s)
        t.text = [f"<b>{money(v)}</b>" if k == chosen else money(v) for k, v in s.items()]
        b.marker.color = [ORANGE if k == chosen else GRAY for k in s.index]

near = lambda v: "about the same as" if abs(v) < 1000 else f"${abs(v) / 1000:,.0f}k {'above' if v > 0 else 'below'}"

def draw1(*_):
    kind, s, pick = c_kind.value, focus[c_kind.value], c_pick.value
    draw_bars(fig1, ranked(s, pick), {k: m.sum() for k, m in masks[kind].items()}, pick)
    say1.value = (f'<p class="say"><b>{pick}</b> jobs start at {money(s[pick])}, {near(s[pick] - START)} the typical start. '
                  f'The best and worst differ by {money(np.ptp(s.to_numpy()))}, and random chance alone gives about {money(chance[kind])}, so this ranking could be luck.</p>')

def draw2(*_):
    d = place(entry)
    g = d.groupby("company_location")["salary_usd"].agg(["median", "size"]).sort_values("median", ascending=False)
    s, c = g["median"], c_country.value
    draw_bars(fig2, ranked(s, c), g["size"], c)
    say2.value = f'<p class="say"><b>{c}</b> starts at {money(s[c])}, {near(s[c] - START)} the typical start. It ranks {list(s.index).index(c) + 1} of {len(s)}.</p>'

def path():
    d = place(df)
    return d[d["company_location"] == c_country.value].groupby("experience")["salary_usd"].median()[LEVELS]

def draw3(*_):
    mine, yr = path(), c_years.value
    now = mine[level_at(yr)]
    with fig3.batch_update():
        for t, p in ((fig3.data[0], by_level), (fig3.data[1], mine)):
            t.x, t.y = YEAR_STEPS, list(p.values) + [p.iloc[-1]]
        m = fig3.data[2]
        m.x, m.y, m.text, m.textposition = [yr], [now], [f"<b>{money(now)}</b>"], "top right" if yr < 2 else "top left"
        ends = [mine["Executive"], by_level["Executive"]]
        lab = fig3.data[3]
        lift = .045 * max(mine.max(), by_level.max()) * (1 if ends[0] >= ends[1] else -1)
        lab.y, lab.text = [ends[0] + lift, ends[1] - lift], [f"<b>{c_country.value}</b>", "All countries"]
        lab.textposition = "middle left"
        fig3.update_yaxes(range=[0, max(mine.max(), by_level.max()) * 1.22])
    gain = "no change from your start" if now == mine["Entry"] else f"{now / mine['Entry']:.1f} times your starting pay"
    say3.value = f'<p class="say">After <b>{yr} {"year" if yr == 1 else "years"}</b> you\'re at {level_at(yr).lower()} level: {money(now)}, {gain}.</p>'

def draw_result(*_):
    mine, yr, pick = path(), c_years.value, c_pick.value
    size = "a company of any size" if c_size.value == "Any size" else f"a {c_size.value.lower()} company"
    f, p, e = focus[c_kind.value][pick] - START, mine["Entry"] - START, mine[level_at(yr)] - mine["Entry"]
    result.value = f"""<div class="res">
<p class="path"><b>Your path: {money(mine['Entry'])} to {money(mine[level_at(yr)])}</b>{pick}, at {size} in {the(c_country.value)}, over {yr} {"year" if yr == 1 else "years"}.</p>
<p class="n"><b>{delta(f)}</b><span>your {c_kind.value.lower()}, within chance</span></p>
<p class="n"><b>{delta(p)}</b><span>your place, vs. a typical start</span></p>
<p class="n"><b>{delta(e)}</b><span>your experience, vs. your start</span></p>
<p class="ans"><b>Overall, experience pays most</b>Where you apply comes second. Which skill, role or industry you pick barely registers.</p></div>"""

# ---------- wiring ----------
def new_kind(*_):
    c_pick.unobserve(redraw_focus, names="value")  # change options and value as one step
    c_pick.description = f"Your {c_kind.value.lower()}"
    c_pick.options = list(masks[c_kind.value])
    c_pick.value = focus[c_kind.value].index[0]
    c_pick.observe(redraw_focus, names="value")
    redraw_focus()

def redraw_focus(*_):
    draw1(); draw_result()

def redraw_place(*_):
    draw2(); draw3(); draw_result()

def redraw_years(*_):
    draw3(); draw_result()

c_kind.observe(new_kind, names="value")
c_pick.observe(redraw_focus, names="value")
for w in (c_country, c_size): w.observe(redraw_place, names="value")
c_years.observe(redraw_years, names="value")
draw1(); draw2(); draw3(); draw_result()

# Convert to live widgets only now: Voila's front end sees a widget's initial state,
# so figures changed after becoming a FigureWidget but before display render blank.
fig1, fig2, fig3 = [go.FigureWidget(f) for f in (fig1, fig2, fig3)]
for f in (fig1, fig2, fig3):
    f._config = {**f._config, "displayModeBar": False, "responsive": True}
    f.add_class("plot")

def choose(control):
    def handler(trace, points, state):
        if points.point_inds: control.value = trace.y[points.point_inds[0]]
    return handler
fig1.data[0].on_click(choose(c_pick))
fig2.data[0].on_click(choose(c_country))

def step(n, title, lead, *parts):
    b = W.VBox([W.HTML(f'<h2><i>{n}</i>{title}</h2><p class="lead">{lead}</p>'), *parts])
    b.add_class("step")
    return b

story = W.Box([opening,
    step(1, "Choose your focus", "Pick what you'd build your résumé around.", c_kind, c_pick, say1, fig1, insight1),
    step(2, "Choose your place", "Now keep the job and change where you apply.", c_country, c_size, say2, fig2, insight2),
    step(3, "Build experience", "Then move time forward in the place you chose.", c_years, say3, fig3, insight3),
    result])
story.add_class("story")
display(story)
# Charts can draw before the page layout settles. Nudge Plotly until each one fits its space (gives up after 60s).
display(Javascript("""
let tries = 0;
const timer = setInterval(() => {
  const off = [...document.querySelectorAll('.js-plotly-plot')].some(p => p._fullLayout &&
    (Math.abs(p._fullLayout.width - p.clientWidth) > 1 || Math.abs(p._fullLayout.height - p.clientHeight) > 1));
  if (off) window.dispatchEvent(new Event('resize'));
  if (++tries > 120) clearInterval(timer);
}, 500);
"""))